# 12-lead ECG image → 4-class LVEF category: CNN comparison on a free Colab GPU

**Before running:** **Runtime → Change runtime type → T4 GPU → Save**, then **Runtime → Run all**.

What this notebook does (same patient-grouped split and the same 210 test ECGs as the README):
1. downloads the project (code + ECG PDFs) from GitHub;
2. builds the ECG images with **7 preprocessing variants** (grid removal, grayscale, Gaussian, median, bilateral, CLAHE, adaptive threshold);
3. **Part A – architectures:** fine-tunes ImageNet-pretrained ResNet-18/34/50, VGG16, DenseNet-121, EfficientNet-B0, MobileNetV2, InceptionV3 and ConvNeXt-Atto on the grid-removed images;
4. **Part B – filters:** trains the same CNN (ConvNeXt-Atto, pretrained) once per preprocessing variant, so only the filter differs;
5. re-fits the header model and fuses it with the best pretrained CNN;
6. prints the comparison tables (accuracy, macro precision / recall / F1, ROC-AUC) and downloads the results.

Already ran the earlier version of this notebook? Upload that `ecg_results.zip` in step 3 and finished models are skipped.

Rough time estimate on a T4 (not measured): 1–2 hours from scratch; less when reusing earlier results.

In [ ]:
# 1. Check that a GPU is attached (should print e.g. 'GPU 0: Tesla T4')
!nvidia-smi -L

In [ ]:
# 2. Get the code and data, install the extra libraries
!git clone --depth 1 -b claude/funny-gauss-udlmv0 https://github.com/krithikareddy22/ECG-Image-Based-Prediction-EF-hear-attack-12-Lead.git ecg
%cd ecg
!pip install -q timm pymupdf scikit-image tabulate

In [ ]:
# 3. (Optional) reuse results from an earlier Colab run: set True, run, choose ecg_results.zip
REUSE_PREVIOUS_RESULTS = False
if REUSE_PREVIOUS_RESULTS:
    from google.colab import files
    for name in files.upload():
        !unzip -o -q "{name}" -d .
    !ls results | grep pretrained

In [ ]:
# 4. Render the ECG PDFs and build all 7 image variants (reuses results/split.csv)
!python -m ecg_ef.preprocess --variants all

## Part A – CNN architectures (ImageNet-pretrained, grid-removed images)
Each model downloads its ImageNet weights automatically and is fine-tuned with lr 3e-4.
A model whose `results/<name>_filtered_pretrained/metrics.json` already exists is skipped.

In [ ]:
import pathlib, torch
print('CUDA available:', torch.cuda.is_available())
ARCHITECTURES = ['resnet18', 'resnet34', 'resnet50', 'vgg16', 'densenet121',
                 'efficientnet_b0', 'mobilenetv2', 'inception_v3', 'convnext_atto']
for m in ARCHITECTURES:
    if pathlib.Path(f'results/{m}_filtered_pretrained/metrics.json').exists():
        print('skip (already trained):', m); continue
    !python -m ecg_ef.train --model {m} --pretrained

## Part B – preprocessing filters (same CNN, only the filter changes)

In [ ]:
FILTER_MODEL = 'convnext_atto'      # best image-only model in the previous Colab run
for v in ['raw', 'gaussian', 'median', 'bilateral', 'clahe', 'adaptive']:
    if pathlib.Path(f'results/{FILTER_MODEL}_{v}_pretrained/metrics.json').exists():
        print('skip (already trained):', v); continue
    !python -m ecg_ef.train --model {FILTER_MODEL} --variant {v} --pretrained

## Header model + fusion with the best pretrained CNN (chosen on the validation set)

In [ ]:
!python -m ecg_ef.header
import json
from ecg_ef.train import selection_score
tags = [p.parent.name for p in pathlib.Path('results').glob('*_pretrained/metrics.json')]
val = {t: selection_score(json.load(open(f'results/{t}/metrics.json'))['val']) for t in tags}
best = max(val, key=val.get)
print('best pretrained CNN on validation:', best, round(val[best], 3))
!python -m ecg_ef.ensemble --members header_logreg {best} --tune-weight --name fusion_header_pretrained

## Comparison (all models, same 210 test ECGs)

In [ ]:
!python -m ecg_ef.compare > compare.log
import pandas as pd
from IPython.display import Image, display
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 30)
d = pd.read_csv('results/comparison.csv')
cols = ['Model', 'Accuracy', 'Macro precision', 'Macro recall', 'Macro-F1', 'Macro AUROC (OvR)',
        'AUROC Normal', 'AUROC Mild', 'AUROC Moderate', 'AUROC Severe', 'HF AUC']
arch = d[d.Model.str.contains('ImageNet') & ~d.Model.str.contains('filter')]
print('Part A - architectures (pretrained, grid-removed images)')
display(arch[cols].sort_values('Macro-F1', ascending=False).round(3))
filt = d[d.Model.str.startswith('ConvNeXt-Atto') & d.Model.str.contains('ImageNet')]
print('Part B - preprocessing filters (ConvNeXt-Atto, pretrained)')
display(filt[cols].sort_values('Macro-F1', ascending=False).round(3))
print('All models:')
display(d[cols + ['Input']].round(3))
display(Image('results/comparison.png'))
print(open('results/best_model.json').read())

## Download the results
Zip of `results/` and the trained models (the 0.5 GB VGG16 model file is left out). Keep it: Colab deletes everything when the session ends.

In [ ]:
!zip -qr ecg_results.zip results models -x 'models/vgg16*'
!ls -lh ecg_results.zip
from google.colab import files
files.download('ecg_results.zip')

## (Optional) Predict your own ECG PDFs with the selected model

In [ ]:
from google.colab import files
uploaded = files.upload()          # choose one or more ECG PDF / PNG / JPG files
for name in uploaded:
    !python -m ecg_ef.predict "{name}"